# AEGIS LLM Orchestration Spike Notebook
=========================================
This exploratory notebook prototypes an LLM-driven detector-selection planner for the AEGIS deepfake detection framework.

> **Note:** This notebook is explicitly a spike and is NOT integrated into the live pipeline or `docker-compose.yml`.

## 1. Static Capability Manifest
Each detector is described by its modality, expected input format, latency (pulled from Month 1 baseline report), and functional description.

In [1]:
import json

CAPABILITY_MANIFEST = [
    {
        "name": "video-classifier",
        "modality": "video",
        "expected_input": "Video frames / MP4 file",
        "approx_latency_ms": 5142,
        "description": "Frame-level EfficientNet-B0 visual spatial deepfake classifier detecting facial manipulation artifacts."
    },
    {
        "name": "aasist",
        "modality": "audio",
        "expected_input": "FLAC / WAV / MP4 audio track",
        "approx_latency_ms": 212,
        "description": "Raw-waveform spectral audio spoofing and voice clone detector using ClovaAI AASIST model."
    },
    {
        "name": "rppg",
        "modality": "video",
        "expected_input": "MP4 facial video (duration >= 2.0s, visible face skin region)",
        "approx_latency_ms": 4507,
        "description": "Physiological rPPG heartbeat-consistency detector using spatial-temporal CHROM algorithm."
    },
    {
        "name": "syncnet",
        "modality": "video+audio",
        "expected_input": "MP4 video file with active audio track and visible mouth ROI",
        "approx_latency_ms": 6505,
        "description": "Audio-visual lip-sync correlation detector evaluating mouth movement alignment with extracted audio MFCC features."
    }
]

print(json.dumps(CAPABILITY_MANIFEST, indent=2))

## 2. Rule-Based vs LLM Detector Selection Runner

In [ ]:
import os
import sys
import csv
import requests
from pathlib import Path

BASE_DIR = Path(".").resolve().parent.parent if Path("orchestrator").exists() == False else Path(".").resolve()
sys.path.insert(0, str(BASE_DIR / "orchestrator" / "app"))

from models import InputMetadata
from rules import decide_detectors_to_call

OPENROUTER_API_KEY = os.environ.get(
    "OPENROUTER_API_KEY",
    "insert key here"
)
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
DEFAULT_MODELS = [
    "nvidia/nemotron-3.5-lightning:free",
    "nvidia/nemotron-3-ultra-550b-a55b:free",
    "inclusionai/ling-3.0-flash-fin:free"
]

def extract_sample_metadata(sample_path: Path, row_modality: str) -> dict:
    path_str = str(sample_path)
    ext = sample_path.suffix.lower()
    file_size = sample_path.stat().st_size if sample_path.exists() else 0
    filename = sample_path.name

    if row_modality == "video" or ext in [".mp4", ".avi", ".mov", ".mkv"]:
        has_audio = False
        try:
            import imageio_ffmpeg, subprocess
            ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()
            cmd = [ffmpeg_exe, "-hide_banner", "-i", path_str]
            res = subprocess.run(cmd, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=2)
            if "Audio:" in res.stderr:
                has_audio = True
        except Exception:
            has_audio = False

        return {
            "filename": filename,
            "modality": "video",
            "has_audio": has_audio,
            "duration_seconds": 12.5 if has_audio else 10.0,
            "resolution": [1920, 1080],
            "file_size_bytes": file_size,
        }
    else:
        return {
            "filename": filename,
            "modality": "audio",
            "has_audio": True,
            "duration_seconds": 4.5,
            "resolution": None,
            "file_size_bytes": file_size,
        }

def call_llm_detector_selection(metadata: dict) -> dict:
    prompt = (
        "You are an intelligent detector-selection planner for AEGIS, a deepfake verification system.\n\n"
        "CAPABILITY MANIFEST:\n"
        f"{json.dumps(CAPABILITY_MANIFEST, indent=2)}\n\n"
        "SAMPLE METADATA:\n"
        f"{json.dumps(metadata, indent=2)}\n\n"
        "INSTRUCTIONS:\n"
        "1. Select ONLY from valid detector names: 'video-classifier', 'aasist', 'rppg', 'syncnet'.\n"
        "2. Do NOT select detectors if their input requirements are violated (e.g. do NOT choose syncnet or aasist if has_audio is False; do NOT choose video detectors if modality is audio).\n"
        "3. Provide your response strictly as raw JSON with no conversational text or markdown code fences, matching this schema:\n"
        "{\n"
        '  "chosen_detectors": ["detector1", "detector2"],\n'
        '  "reasoning": "Brief explanation of selection and exclusions."\n'
        "}\n"
    )
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}", "Content-Type": "application/json"}
    for model_name in DEFAULT_MODELS:
        payload = {"model": model_name, "messages": [{"role": "user", "content": prompt}], "temperature": 0.1}
        try:
            resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=4)
            if resp.status_code == 200:
                raw_text = resp.json()["choices"][0]["message"]["content"].strip()
                clean_text = raw_text
                if clean_text.startswith("```"):
                    clean_text = clean_text.split("```", 2)[1]
                    if clean_text.startswith("json"): clean_text = clean_text[4:]
                    clean_text = clean_text.strip()
                parsed = json.loads(clean_text)
                return {"chosen_detectors": parsed.get("chosen_detectors", []), "reasoning": parsed.get("reasoning", ""), "model": model_name}
        except Exception:
            continue
    return {"chosen_detectors": ["video-classifier", "rppg"] if metadata["modality"] == "video" else ["aasist"], "reasoning": "Fallback rule", "model": "fallback"}

## 3. Side-by-Side Comparison Output

In [3]:
manifest_file = BASE_DIR / "eval" / "data" / "eval_manifest.csv"
with open(manifest_file, newline="", encoding="utf-8") as f:
    reader = list(csv.DictReader(f))

v_fakes = [s for s in reader if s["modality"] == "video" and s["true_label"] == "fake"][:3]
v_reals = [s for s in reader if s["modality"] == "video" and s["true_label"] == "real"][:2]
a_fakes = [s for s in reader if s["modality"] == "audio" and s["true_label"] == "fake"][:3]
a_reals = [s for s in reader if s["modality"] == "audio" and s["true_label"] == "real"][:2]
sample_batch = v_fakes + v_reals + a_fakes + a_reals

print(f"{'#':<3} | {'Sample Name':<38} | {'Modality':<8} | {'Rule-Based Choice':<30} | {'LLM Choice':<30}")
print("-" * 115)
for idx, s in enumerate(sample_batch, 1):
    abs_path = BASE_DIR / s["sample_path"]
    meta = extract_sample_metadata(abs_path, s["modality"])
    input_meta = InputMetadata(filename=meta["filename"], modality=meta["modality"], has_audio=meta["has_audio"], duration_seconds=meta["duration_seconds"], resolution=meta["resolution"])
    rb_choice = ",".join(decide_detectors_to_call(input_meta))
    llm_res = call_llm_detector_selection(meta)
    llm_choice = ",".join(llm_res["chosen_detectors"])
    print(f"{idx:<3} | {meta['filename']:<38} | {meta['modality']:<8} | {rb_choice:<30} | {llm_choice:<30}")